In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

from tqdm import tqdm

from medmnist import TissueMNIST
from medmnist import INFO

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("CUDA is not available. Using CPU.")

In [ ]:
info = INFO['tissuemnist']
data_flag = 'tissuemnist'
DataClass = TissueMNIST

task = info['task']  
n_channels = info['n_channels']
n_classes = len(info['label'])

print(f"Number of classes:", n_classes)
print(f"Number of channels: {n_channels}")

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  
    transforms.RandomRotation(degrees=15),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
    lambda x: x.unsqueeze(0)
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
    lambda x: x.unsqueeze(0)
])

In [ ]:
data_train = DataClass(split='train', transform=train_transform, download=True, size=128)
data_test = DataClass(split='test', transform=eval_transform, download=True, size=128)
data_eval = DataClass(split='val', transform=eval_transform, download=True, size=128)

In [ ]:
batch_size = 32

dataloader_train = DataLoader(dataset=data_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset=data_test, batch_size=batch_size, shuffle=False)
dataloader_eval = DataLoader(dataset=data_eval, batch_size=batch_size, shuffle=False)

print(f"\nNumber of images in training dataset: {len(data_train)}")
print(f"Number of images in test dataset: {len(data_test)}")
print(f"Number of images in validation dataset: {len(data_eval)}")


In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
criterion = nn.BCELoss().to(device)
epochs = 10

In [ ]:
best_model_path = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/notebooks/classical/best_model.pth"

best_val_accuracy = 0.0  

train_losses = []
val_losses = []
val_accuracies = []
val_precisions = []
val_recalls = []
val_f1_scores = []
val_aucs = []

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")

    model.train()
    total_loss = 0.0
    print("\n[Training]")
    for batch_idx, (images, labels) in enumerate(tqdm(dataloader_train, desc="Training Batches", bar_format="{desc}: {n}/{total}")):
        images, labels = images.squeeze(1).to(device), labels.squeeze().to(device)

        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    epoch_train_loss = total_loss / len(dataloader_train)
    train_losses.append(epoch_train_loss)
    print(f"Epoch {epoch + 1} Training Loss: {epoch_train_loss:.6f}")

    scheduler.step()

    model.eval()
    val_loss = 0.0
    val_labels, val_predictions = [], []

    print("\n[Validation]")
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(tqdm(dataloader_eval, desc="Validation Batches", bar_format="{desc}: {n}/{total}")):
            images, labels = images.squeeze(1).to(device), labels.squeeze().to(device)
            output = model(images)
            loss = criterion(output, labels)
            val_loss += loss.item()

            val_labels.append(labels)
            val_predictions.append(output)

    epoch_val_loss = val_loss / len(dataloader_eval)
    val_losses.append(epoch_val_loss)
    val_labels = torch.cat(val_labels)
    val_predictions = torch.cat(val_predictions)

    val_accuracy = accuracy_score(
        val_labels.cpu().numpy(), val_predictions.argmax(dim=1).cpu().numpy())
    val_precision = precision_score(
        val_labels.cpu().numpy(), val_predictions.argmax(dim=1).cpu().numpy(),
        average="weighted", zero_division=0)
    val_recall = recall_score(
        val_labels.cpu().numpy(), val_predictions.argmax(dim=1).cpu().numpy(),
        average="weighted", zero_division=0)
    val_f1 = f1_score(
        val_labels.cpu().numpy(), val_predictions.argmax(dim=1).cpu().numpy(),
        average="weighted", zero_division=0)
    val_auc = roc_auc_score(
        val_labels.cpu().numpy(), val_predictions[:, 1].cpu().numpy())

    val_accuracies.append(val_accuracy)
    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1_scores.append(val_f1)
    val_aucs.append(val_auc)

    print(
        f"\nEpoch {epoch + 1} Summary:\n"
        f"Train Loss: {epoch_train_loss:.6f}, "
        f"Val Loss: {epoch_val_loss:.6f}, "
        f"Accuracy: {val_accuracy:.6f}, "
        f"Precision: {val_precision:.6f}, "
        f"Recall: {val_recall:.6f}, "
        f"F1: {val_f1:.6f}, "
        f"AUC: {val_auc:.6f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved with accuracy: {best_val_accuracy:.3f}")


In [ ]:
# model_path = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/notebooks/classical/best_model.pth"
# model.load_state_dict(torch.load(model_path, weights_only=True))

test_loss = 0.0
test_labels, test_predictions = [], []

model.eval()
with torch.no_grad():
    for images, labels in dataloader_test:
        images, labels = images.squeeze(1).to(device), labels.squeeze().to(device)
        output = model(images)
        loss = criterion(output, labels)
        test_loss += loss.item()
        test_labels.append(labels)
        test_predictions.append(output)

test_labels = torch.cat(test_labels)
test_predictions = torch.cat(test_predictions)

test_probs = torch.exp(test_predictions)

test_accuracy = accuracy_score(
    test_labels.cpu().numpy(), test_predictions.argmax(dim=1).cpu().numpy()
)
test_precision = precision_score(
    test_labels.cpu().numpy(), test_predictions.argmax(dim=1).cpu().numpy(), 
    average="weighted", zero_division=0
)
test_recall = recall_score(
    test_labels.cpu().numpy(), test_predictions.argmax(dim=1).cpu().numpy(), 
    average="weighted", zero_division=0
)
test_f1 = f1_score(
    test_labels.cpu().numpy(), test_predictions.argmax(dim=1).cpu().numpy(), 
    average="weighted", zero_division=0
)
test_auc = roc_auc_score(
    test_labels.cpu().numpy(), test_probs[:, 1].cpu().numpy()
)

print("\nFinal Test Evaluation:")
print(f"Test Loss: {test_loss / len(dataloader_test):.6f}")
print(f"Test Accuracy: {test_accuracy:.6f}")
print(f"Test Precision: {test_precision:.6f}")
print(f"Test Recall: {test_recall:.6f}")
print(f"Test F1 Score: {test_f1:.6f}")
print(f"Test AUC: {test_auc:.6f}")

In [ ]:
best_model_path = "best_model.pth"

model.load_state_dict(torch.load(best_model_path))
model = torch.nn.Sequential(*list(model.children())[:-1])  
model.to(device)
model.eval()

def extract_features(dataloader, dataset_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    features_list, labels_list = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Extracting Features ({dataset_name})"):
            images = images.to(device)
            features = model(images)
            features_list.append(features.cpu().numpy())
            labels_list.append(labels.cpu().numpy())

    features_array = np.vstack(features_list)
    labels_array = np.hstack(labels_list)

    np.save(os.path.join(output_dir, f"features_{dataset_name}.npy"), features_array)
    np.save(os.path.join(output_dir, f"labels_{dataset_name}.npy"), labels_array)

    df = pd.DataFrame(features_array)
    df['label'] = labels_array
    df.to_csv(os.path.join(output_dir, f"features_{dataset_name}.csv"), index=False)

    print(f"{dataset_name} features saved in {output_dir}.")

output_directory = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/classical"
extract_features(dataloader_train, "MedMNIST", output_directory)